## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


1. make sure the pipeline is the same, the order of the tools should remain the same
2. make sure the version of the softwares and tools are the same, best use the same environment
3. make sure the parameters are the same, better provide documentation of it

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

Quality control and preprocessing of the data, use nf-core/rnaseq.

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

In [7]:
import os
import numpy as np
import pandas as pd

# Load metadata from Excel sheet
meta = pd.read_excel("conditions_runs_oxy_project.xlsx")
meta.columns = meta.columns.str.strip()
meta = pd.DataFrame({
    "run": meta["Run"].astype(str).str.strip(),
    "condition": np.where(meta["Condition: Oxy"].notna(), "Oxy", "Sal"),
    "surgery": np.where(meta["Genotype: SNI"].notna(), "SNI", "Sham"),
})

# FASTQ paths from fetchngs (or tutor's fetchngs_out folder)
fq = pd.read_csv("fetchngs_out/samplesheet/samplesheet.csv")
fq = fq.merge(meta, left_on="run_accession", right_on="run")

# Construct the nf-core/rnaseq samplesheet
samplesheet = pd.DataFrame({
    "sample": fq["surgery"] + "_" + fq["condition"] + "_" + fq["run"],
    "fastq_1": fq["fastq_1"].apply(os.path.abspath),
    "fastq_2": fq["fastq_2"].apply(os.path.abspath),
    "strandedness": "auto",
})
samplesheet.to_csv("samplesheet_rnaseq.csv", index=False)
samplesheet

,sample,fastq_1,fastq_2,strandedness
0,SNI_Oxy_SRR23195516,/Users/ivan/Desktop/CompWorkflow/computational...,/Users/ivan/Desktop/CompWorkflow/computational...,auto
1,Sham_Oxy_SRR23195511,/Users/ivan/Desktop/CompWorkflow/computational...,/Users/ivan/Desktop/CompWorkflow/computational...,auto


In [11]:


!nextflow run nf-core/rnaseq \
    -r 3.27.0 \
    -profile docker \
    --input samplesheet_rnaseq.csv \
    --outdir results/rnaseq \
    --genome GRCm38 \
   


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/rnaseq` [romantic_kay] revision: a1fcdddd3b [3.27.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input              : samplesheet_rnaseq.csv
  outdir             : results/rnaseq

Reference genome options
  genome             : GRCm38
  fasta              : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Sequence/WholeGenomeFasta/genome.fa
  gtf                : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
  gene_bed           : s3://ngi-igenomes/igenomes//Mu

Run locally .nf-core/rnaseq, It failed at MAKE_TRANSCRIPTS_FASTA because the step required 72 GB of RAM and only 16 GB were available. The analysis below uses the tutor-provided results in rnaseq_out_allsamples

In [1]:
# chheck out thhe pipeline info e.g. the versions
!ls rnaseq_out_allsamples/pipeline_info rnaseq_out_allsamples/star_salmon rnaseq_out_allsamples/multiqc/*

rnaseq_out_allsamples/multiqc/star_salmon:
multiqc_report.html  multiqc_report_data  multiqc_report_plots

rnaseq_out_allsamples/pipeline_info:
execution_report_2026-09-29_16-00-28.html
execution_report_2026-09-29_16-02-58.html
execution_report_2026-09-29_16-04-08.html
execution_report_2026-09-29_16-04-56.html
execution_report_2026-09-29_16-10-23.html
execution_timeline_2026-09-29_16-00-28.html
execution_timeline_2026-09-29_16-02-58.html
execution_timeline_2026-09-29_16-04-08.html
execution_timeline_2026-09-29_16-04-56.html
execution_timeline_2026-09-29_16-10-23.html
execution_trace_2026-09-29_16-00-28.txt
execution_trace_2026-09-29_16-02-58.txt
execution_trace_2026-09-29_16-04-08.txt
execution_trace_2026-09-29_16-04-56.txt
execution_trace_2026-09-29_16-10-23.txt
nf_core_rnaseq_software_mqc_versions.yml
params_2026-09-29_16-00-40.json
params_2026-09-29_16-03-08.json
params_2026-09-29_16-04-18.json
params_2026-09-29_16-05-07.json
params_2026-09-29_16-10-34.json
pipeline_dag_2026-09-29_1

nf-core/rnaseq takes RNA reads, conduct quality trimming, and turns them into gene expression counts by alignment to reference genome.

https://nf-co.re/rnaseq/3.14.0/
Main tools in the pipeline:
FastQC: raw read quality
Trim Galore: adapter and quality trimming
SortMeRNA: rRNA removal, only if you set --remove_ribo_rna
STAR + Salmon: alignment, then gene/transcript quantification (default star_salmon route)
SAMtools, Picard MarkDuplicates: BAM processing and duplicate marking
RSeQC, Qualimap, dupRadar, Preseq: RNA-seq-specific QC (read distribution, strandedness, gene body coverage, duplication vs. expression, library complexity)
featureCounts biotype QC: includes the fraction of reads on rRNA genes
MultiQC: combines everything into one report 


As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [ ]:
# post here the command you used to run nf-core/rnaseq

Explain all the parameters you set and why you set them in this way.



-r 3.27.0	#pipeline version, be specific for reproducibility.
-profile docker	#Every tool runs in a fixed container, so the software environment is identical on any machine.
--input samplesheet_rnaseq.csv	#specify input data
--outdir results/rnaseq	 #create folder for output
--genome GRCm38	#The mouse reference genome and annotation from iGenomes, because the samples are mouse.


## Browsing the results

How did the pipeline perform?

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

High values in these factor can lead to poor results because:

- ribosomal rRNA: hihgh in GC, high rRNA means we have less useful mRNA
- Duplication: too many duplications shows less genes are covered.
- GC content: indicating conotaminaton from rRNA

Would you exclude any samples? If yes, which and why?

In [ ]:
"SRX19144492",  
26% uniquely mapped, only 7.3 M aligned reads, 63% GC, 87% adapter

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

Try to differentiate expression levels of genes.